In [11]:
import os
import pickle
import torch
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms, models
from torchvision.models import Inception_V3_Weights
from PIL import Image
import logging
import sys

# ----------------------------
# Configuration and Setup
# ----------------------------

# Configure logging to display timestamps and log levels
logging.basicConfig(
    stream=sys.stdout,
    level=logging.INFO,  # Set to DEBUG for more detailed logs
    format='%(asctime)s - %(levelname)s - %(message)s'
)
logger = logging.getLogger()

# ----------------------------
# Dataset Definition
# ----------------------------


class PolyvoreDataset(Dataset):
    """
    Custom Dataset for loading Polyvore images.

    Args:
        root_dir (str): Root directory containing category subdirectories.
        transform (callable, optional): Transformations to apply to images.
        max_images_per_class (int, optional): Maximum number of images to load per category.
    """

    def __init__(self, root_dir, transform=None, max_images_per_class=100):
        self.root_dir = root_dir
        self.transform = transform
        self.image_paths = []
        self.labels = []
        self.classes = sorted([
            d for d in os.listdir(root_dir)
            if os.path.isdir(os.path.join(root_dir, d))
        ])
        self.class_to_idx = {cls_name: idx for idx,
                             cls_name in enumerate(self.classes)}

        logger.info(f"Found classes: {self.classes}")

        for cls_name in self.classes:
            cls_folder = os.path.join(root_dir, cls_name)
            images = [
                img_name for img_name in os.listdir(cls_folder)
                if img_name.lower().endswith(('.jpg', '.jpeg', '.png'))
            ]
            selected_images = images[:max_images_per_class]
            logger.info(
                f"Processing class '{cls_name}' with {len(selected_images)} images.")
            for img_name in selected_images:
                relative_path = os.path.join(root_dir, cls_name, img_name)
                if os.path.exists(relative_path):
                    self.image_paths.append(relative_path)
                    self.labels.append(self.class_to_idx[cls_name])
                else:
                    logger.warning(
                        f"Image not found at path: {relative_path}. Skipping.")

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        """
        Returns:
            tuple: (image_tensor, label, image_path)
        """
        img_path = self.image_paths[idx]
        label = self.labels[idx]
        try:
            image = Image.open(img_path).convert('RGB')
            if self.transform:
                image = self.transform(image)
            return image, label, img_path
        except Exception as e:
            logger.error(
                f"Error loading image {img_path}: {e}. Returning a black image.")
            # Return a dummy image and label to prevent DataLoader from crashing
            dummy_image = torch.zeros(3, 299, 299)
            return dummy_image, label, img_path

# ----------------------------
# Feature Extraction Function
# ----------------------------


def extract_and_store_features(model, loader, feature_file='polyvore_features.pkl'):
    """
    Extracts features from images and stores them in a pickle file.

    Args:
        model (torch.nn.Module): Pre-trained model for feature extraction.
        loader (DataLoader): DataLoader for the dataset.
        feature_file (str, optional): Filename to save the features.
    """
    all_features = []
    all_labels = []
    all_image_paths = []
    all_classes = loader.dataset.classes
    total_batches = len(loader)
    logger.info(f"Starting feature extraction over {total_batches} batches.")

    for batch_idx, batch in enumerate(loader):
        if len(batch) != 3:
            logger.error(
                f"Batch {batch_idx + 1} does not contain 3 elements. Found {len(batch)} elements.")
            continue  # Skip this batch

        images, labels, img_paths = batch
        logger.info(
            f"Processing batch {batch_idx + 1}/{total_batches} with {len(images)} images.")

        # Log sample image paths (for debugging purposes)
        logger.debug(
            f"Sample image paths in batch {batch_idx + 1}: {img_paths[:2]}")

        try:
            with torch.no_grad():
                features = model(images)  # Extract features
            all_features.append(features.cpu())
            all_labels.extend(labels.tolist())
            all_image_paths.extend(img_paths)
        except Exception as e:
            logger.error(
                f"Error during feature extraction in batch {batch_idx + 1}: {e}")

    if all_features:
        all_features = torch.cat(all_features, dim=0)
        with open(feature_file, 'wb') as f:
            pickle.dump({
                'features': all_features,
                'labels': all_labels,
                'image_paths': all_image_paths,
                'classes': all_classes
            }, f)
        logger.info(f"Features successfully saved to '{feature_file}'")
    else:
        logger.error("No features were extracted. Feature file not created.")

# ----------------------------
# Dataset Validation Function
# ----------------------------


def validate_dataset(dataset):
    """
    Validates that all image paths in the dataset exist.

    Args:
        dataset (PolyvoreDataset): The dataset to validate.
    """
    logger.info("Validating dataset integrity...")
    missing_images = []
    for idx, img_path in enumerate(dataset.image_paths):
        if not os.path.exists(img_path):
            logger.warning(f"Missing image at path: {img_path}")
            missing_images.append(idx)

    if missing_images:
        logger.error(
            f"Found {len(missing_images)} missing images in the dataset.")
    else:
        logger.info("All image paths are valid.")

# ----------------------------
# DataLoader Testing Function
# ----------------------------


def test_data_loader(loader):
    """
    Tests the DataLoader by fetching the first batch and displaying its contents.

    Args:
        loader (DataLoader): The DataLoader to test.
    """
    logger.info("Testing DataLoader...")
    try:
        batch = next(iter(loader))
        if len(batch) != 3:
            logger.error(
                f"Test batch does not contain 3 elements. Found {len(batch)} elements.")
            return False
        images, labels, img_paths = batch
        logger.info(f"Test batch contains {len(images)} images.")
        logger.info(f"Sample image path: {img_paths[0]}")
        return True
    except Exception as e:
        logger.error(f"Error during DataLoader testing: {e}")
        return False

# ----------------------------
# Main Execution
# ----------------------------


def main():
    # Define transformations for the images
    transform = transforms.Compose([
        # Resize to match InceptionV3 input size
        transforms.Resize((299, 299)),
        transforms.ToTensor(),
        transforms.Normalize(
            # Normalization parameters for InceptionV3
            mean=[0.485, 0.456, 0.406],
            std=[0.229, 0.224, 0.225]
        )
    ])

    # Specify the root directory of your dataset
    dataset_root_dir = 'Re-PolyVore'  # Replace with your dataset directory
    if not os.path.isdir(dataset_root_dir):
        logger.error(
            f"Dataset directory '{dataset_root_dir}' does not exist. Exiting.")
        return

    # Initialize the dataset and DataLoader
    polyvore_dataset = PolyvoreDataset(
        root_dir=dataset_root_dir,
        transform=transform,
        max_images_per_class=25  # Adjust as needed
    )

    if len(polyvore_dataset) == 0:
        logger.error("No images found in the dataset. Exiting.")
        return

    # Validate dataset integrity
    validate_dataset(polyvore_dataset)

    # Configure DataLoader
    batch_size = 32  # Adjust based on your system's CPU and memory
    num_workers = 0  # Set to 0 for debugging; increase based on your CPU cores

    feature_loader = DataLoader(
        dataset=polyvore_dataset,
        batch_size=batch_size,
        shuffle=False,
        num_workers=num_workers,
        pin_memory=False  # Set to True if using GPU, False for CPU
    )

    # Test the DataLoader
    if not test_data_loader(feature_loader):
        logger.error("DataLoader test failed. Exiting.")
        return

    # Device configuration (CPU only)
    device = torch.device('cpu')
    logger.info(f"Using device: {device}")

    # Load the pre-trained InceptionV3 model
    weights = Inception_V3_Weights.DEFAULT
    try:
        inception = models.inception_v3(
            weights=weights, aux_logits=True)  # Keep aux_logits=True
    except ValueError as ve:
        logger.error(f"ValueError during model initialization: {ve}")
        return
    except Exception as e:
        logger.error(f"Unexpected error during model initialization: {e}")
        return

    # Replace the auxiliary logits with Identity to disable it
    inception.AuxLogits = torch.nn.Identity()
    # Replace the final layer with Identity to get features
    inception.fc = torch.nn.Identity()
    inception.to(device)
    inception.eval()
    logger.info("InceptionV3 model loaded and set to evaluation mode.")

    # Extract and store features
    feature_file = 'polyvore_features.pkl'
    if not os.path.exists(feature_file):
        logger.info(
            f"Feature file '{feature_file}' not found. Starting feature extraction...")
        extract_and_store_features(inception, feature_loader, feature_file)
    else:
        logger.info(
            f"Features already computed and stored in '{feature_file}'.")


if __name__ == "__main__":
    main()

2024-09-28 21:56:28,117 - INFO - Found classes: ['bag', 'bracelet', 'brooch', 'dress', 'earrings', 'eyewear', 'gloves', 'hairwear', 'hats', 'jumpsuit', 'legwear', 'necklace', 'neckwear', 'outwear', 'pants', 'rings', 'shoes', 'skirt', 'top', 'watches']


2024-09-28 21:56:28,132 - INFO - Processing class 'bag' with 25 images.
2024-09-28 21:56:28,139 - INFO - Processing class 'bracelet' with 25 images.
2024-09-28 21:56:28,142 - INFO - Processing class 'brooch' with 25 images.
2024-09-28 21:56:28,147 - INFO - Processing class 'dress' with 25 images.
2024-09-28 21:56:28,153 - INFO - Processing class 'earrings' with 25 images.
2024-09-28 21:56:28,158 - INFO - Processing class 'eyewear' with 25 images.
2024-09-28 21:56:28,161 - INFO - Processing class 'gloves' with 25 images.
2024-09-28 21:56:28,164 - INFO - Processing class 'hairwear' with 25 images.
2024-09-28 21:56:28,165 - INFO - Processing class 'hats' with 25 images.
2024-09-28 21:56:28,169 - INFO - Processing class 'jumpsuit' with 25 images.
2024-09-28 21:56:28,170 - INFO - Processing class 'legwear' with 25 images.
2024-09-28 21:56:28,174 - INFO - Processing class 'necklace' with 25 images.
2024-09-28 21:56:28,176 - INFO - Processing class 'neckwear' with 25 images.
2024-09-28 21:56: